# 06. Avaliação final (teste usado uma única vez)

Roteiro da última etapa do protocolo:

1. desenvolvimento concluído (`run_validation`, notebooks 02, 03 e 05);
2. protocolo congelado no terminal, na raiz do repositório:
   ```
   python -m src.pipeline.freeze
   ```
   que grava `results/protocolo.lock` com os hashes de todos os arquivos versionados em `src/**/*.py` e `data/*.py` e de `data/selecao_datasets.txt`, o espaço de busca, o orçamento e o commit;
3. **lock registrado no git ANTES da avaliação final**:
   ```
   git add results/protocolo.lock
   git commit -m "Congela o protocolo"
   ```
4. **avaliação final única** (`final_test`): cada sistema é ajustado no treino de cada fold externo e avaliado uma vez no teste;
5. **resultados registrados no git IMEDIATAMENTE**:
   ```
   git add results
   git commit -m "Avaliação final"
   ```
6. conferência do histórico: `python -m src.pipeline.check_protocol`.

`final_test` só executa em um repositório git, sem alterações não comitadas em `src/`, `data/` e no lock, com o lock versionado e idêntico ao de `HEAD`, com os hashes conferindo e sem nenhum vestígio de avaliação anterior (`test.csv`, `test_log.csv`, `test_manifest.json`, `preds/` ou commit com resultados). Use `--resume` somente para completar combinações pendentes ou com erro, com o mesmo lock. As travas locais são dissuasórias; a evidência do protocolo é o histórico git. Depois da avaliação final, não altere o modelo, o espaço de busca nem a seleção: o relatório deve apresentar o resultado tal como obtido.

In [ ]:
import sys
from pathlib import Path

REPO = Path('..').resolve()
sys.path.insert(0, str(REPO))

import json

import pandas as pd

from src.pipeline import check_protocol, final_test
from src.pipeline.freeze import LOCK_FILE, lock_digest, read_lock, verify_lock

RESULTS = REPO / 'results'

## 1. Conferir o lock

In [ ]:
lock_path = RESULTS / LOCK_FILE
lock = read_lock(lock_path)
print('sha256 do lock:', lock_digest(lock_path))
print('commit:', lock['git_commit'], '| alterações não registradas:', lock['git_dirty'],
      '| arquivos travados:', len(lock['files']))
print('orçamento:', lock['budget'], '| semente:', lock['seed'])
print('sistemas:', lock['models'], '| folds externos:', lock['folds'], '| datasets:', len(lock['task_ids']))
problems = verify_lock(lock, REPO)
print('Lock íntegro.' if not problems else problems)

## 2. Executar a avaliação final

Antes, confirme que o lock está comitado (`git log -- results/protocolo.lock`). A célula abaixo executa `final_test` (equivalente a `python -m src.pipeline.final_test` na raiz). Ela pode levar horas; prefira executá-la no terminal. **Execute uma única vez** e, em seguida, comite `results/`.

In [ ]:
EXECUTAR = False   # mude para True conscientemente, uma única vez

if EXECUTAR:
    final_test.main(['--output-dir', str(RESULTS)], root=REPO)   # recusa se houver vestígio de avaliação
else:
    print('Avaliação final não executada nesta célula.')

## 3. Ler os resultados, o log e o manifesto

In [ ]:
test = pd.read_csv(RESULTS / 'test.csv')
print(test['status'].value_counts())
test.groupby('model')[['auc_ovo', 'accuracy', 'g_mean', 'cross_entropy', 'total_time_s']].mean()

In [ ]:
display(pd.read_csv(RESULTS / 'test_log.csv'))
manifest = json.loads((RESULTS / 'test_manifest.json').read_text(encoding='utf-8'))
states = pd.Series(manifest['status']).value_counts().to_dict()
print('lock usado:', manifest['lock_sha256'], '| execuções:', len(manifest['runs']),
      '| commits (git_head):', [r['git_head'][:10] for r in manifest['runs']], '| estados:', states)

## 4. Conferir o protocolo pelo histórico git

Depois de comitar `results/`, `check_protocol` deve terminar com **APROVADO**. A mesma verificação é feita na correção.

In [ ]:
print(check_protocol.check_protocol(RESULTS, repo=REPO).text())

A análise estatística (Demšar, bootstrap e regimes) está no notebook 04, que lê `results/test.csv`. Compare, de forma descritiva, a AUC média do teste com a da validação interna (`results/validation.csv`): diferenças grandes indicam otimismo na fase de desenvolvimento e devem ser discutidas no relatório.